INL/CON-26-91533

# Fixing a Radial Build Model for Fusion Applications

## Scenario
* Trying to model a Tokamak fusion power plant
* You have simple, but "buggy", radial build model
* You are trying to perform a parametric sweep

# Background
* Many fusion power concepts use: deuterium ($^2H/D$) - tritium ($^3H/T$) fusion

$$ D+T \to ^4He +n  + 17.6 {\rm MeV}$$

* One technology being considered for a fusion power plant: magnetic confinement
    * Such as a Tokamak 
 


## Example Tokamak

![ARC reactor concept](https://ars.els-cdn.com/content/image/1-s2.0-S0920379615302337-gr1.jpg)

ARC reactor concept from MIT. 
By: [Sorbom, et al.](https://dspace.mit.edu/handle/1721.1/112016), licensed under: [CC-BY-NC-ND-4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/)

# The Need for Tritium Breeding
* Uses deuterium and tritium for fuel
* Deuterium is abundant enough
  * (for example, CANDU reactors use it in large quantities)

## Tritium Pop Quiz
* What's the half-life of tritium?

* No natural source of tritium
* Inventory is constantly decaying
  * Bonus points for calculating the annual loss due to radioactive decay
* Current global supply may supply a *single* power plant for a few years at most
* Must breed tritium onsite

$$ ^6Li + n \to T + ^4He$$

# Quick overview of Major Components Modeled
![exploded view of ARC reactor](https://ars.els-cdn.com/content/image/1-s2.0-S0920379615302337-gr3.jpg)

Exploded view of the ARC reactor. By: [Sorbom, et al.](https://dspace.mit.edu/handle/1721.1/112016), licensed under: [CC-BY-NC-ND-4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/)

* Plasma
* First wall
* Vacuum vessel
* Breeding blanket
* Neutron shielding and other in-vessel components (not modeled)
* Magnets

# Base Model

* 1-D radial build model
* Very simplified and not representative
* Good enough for scoping
* Regions
  * Vacuum
  * First wall (tungsten)
  * Vacuum vessel ("stainless steel")
  * Breeder
  * Void
  * Magnets (REBCO)

# Plot of Geometry
## YZ slice
![YZ plot of the radial build model showing concentric rings of different colors](figs/fusion_radial_YZ.png)

# Quarter Slice of XY slice
![XY plot of the radial build model showing concentric quarter circles](figs/fusion_radial_XY.png)

# First Steps
* Install MontePy
  * This is only needed for jupyter lite
  * Usually `pip install montepy` will persist (also on conda-forge)
* `import montepy`
* Read the model

In [ ]:
# Only needed on jupyter lite
from _config import install_montepy

await install_montepy()

In [ ]:
import montepy

problem = montepy.read_input("models/fusion_tokomak.imcnp")

# Explore the model
* [See the doc strings](https://www.montepy.org/en/stable/api/generated/montepy.MCNP_Problem.html#montepy.MCNP_Problem)

In [ ]:
# prints out a summary of the cells in the problem
print(problem.cells)
# summary of the surfaces
print(problem)
# summary of the materials

# Print the version of MCNP this model is targeting


# Explore Cells
* Let's look at the cells and their comments

In [ ]:
# Cells acts a lot like python dictionaries
#
# Iterate over the cells and their numbers, thanks to items()
for num, cell in problem.cells.items():
    #print the cell number and the cell comments
    print()

# Update First Wall Cell
* Grab the cell by its number (previous) step
* Print the cell density
  * [helpful cell docs](https://www.montepy.org/en/stable/api/generated/montepy.Cell.html#montepy.Cell)


In [ ]:
first_wall = problem.cells[]
print(first_wall.mass_density)

* This is near the theoretical maximum density for tungsten.
* First walls will likely not achieve this.
  * Strong candidate is [tungsten-fiber reinforced tunsgten](https://doi.org/10.1088/1741-4326/ac8c55)
  * Can realistically only achieve around 80% density.

## task: Update mass density to be 80% theoretical 

In [ ]:
first_wall.

## Task 1.2 Check the material definition

* Print the material details
  * `repr(mat)` will be helpful
  * creates a string of all of the nuclides in the materia

In [ ]:
tungsten = 
print(repr(tungsten))

# Update the Material definition
* Tungsten has more than one stable isotope

## Steps
1. Make a dictionary of the stable isotopes using the [2013 IUPAC Isotopic Compositions report](https://doi.org/10.1515/pac-2015-0503)
2. Clear the material
3. Add all nuclides to the material again
   1. [`add_nuclide`](https://www.montepy.org/en/stable/api/generated/montepy.Material.html#add-new-component)
   2. Use library: `82c`, ENDF/B-VII.1 at 900 K

In [ ]:
w_natural = {
    182: 0.265,
    183: 0.143,
    184: 0.306,
    186: 0.284
}
tungsten.clear()
for mass, abundance in w_natural.items():
    tungsten.add_nuclide()


# Update material for Breeder blanket
* Inspect the material in use for the breeder

In [ ]:
breeder = problem.cells[]
breeder_mat = 
print(repr(breeder_mat))

# Issue: Pure lithium metal is not a good breeder

* Alkali metals are challenging to work with and contain
* Pure lithium is very low density (0.5 g/cc)
* Many options exist:
  * $\rm FLiBe$
  * $\rm PbLi$ eutectic
  * $\rm Li_4SiO_4$
  * $\rm PbLi_xO_y$

# Update to use FLiBe
* Starting with FLiBe
* Using [material mixing function](https://www.montepy.org/en/stable/api/generated/montepy.Materials.html#montepy.Materials.mix)
* Need to make elemental nuclides first
   * Bonus Questions: Why should you not use atomic cross sections for neutron transport?
   * Have Li already defined as a material
   * Define [F](https://en.wikipedia.org/wiki/Fluorine) and [Be](https://en.wikipedia.org/wiki/Beryllium) materials
   * Use `82c` library again

In [ ]:
lithium = breeder_mat
fluorine = montepy.Material()
fluorine.add_nuclide("F-19.82c", 1.0)
beryllium = montepy.Material()
beryllium.add_nuclide("Be-9.82c", 1.0)

## Make the FLiBe
* Actually mixture of $\rm LiF$ and $\rm BeF_2$
* Assume stochiometric mixture: $\rm Li_2[BeF_4]$
* Use ARC paper density: 1.94 g/cc

In [ ]:
flibe = problem.materials.mix()
flibe.normalize()
breeder.material = 
breeder.mass_density = 
problem.materials.remove(

# Final task: Perform A Parametric Sweep
* Model has a lot of room for different thicknesses of tritium breeder blankets
* Make multiple models with different thicknesses (cylinder radii)

# Find the outer toroidal surface first bda spell
* Each cell has a [`surfaces`](https://www.montepy.org/en/stable/api/generated/montepy.Cell.html#montepy.Cell.surfaces) attribute
* You can then filter by [surface type mnemonic](https://www.montepy.org/en/stable/api/generated/montepy.Surfaces.html#montepy.Surfaces.cz)

In [ ]:
for torus in breeder.surfaces.tz:
    print()

In [ ]:
breeder_od = breeder.surfaces[]

# Updating breeder can outer cell
* breeder is contained in a can (or cladding)
* Need to update the outer surface of the can to prevent interference
* Can be found by ordering the surfaces by minor radii
  * can add a `key` function to `sorted` to sort by other than the surface number

In [ ]:
key_func = lambda torus: torus.minor_radius

ordered_tori = sorted()
breeder_od_idx = ordered_tori.index(breeder_od)
breeder_can_od = 

# Put it all together   
* Use [`numpy.arange`](https://numpy.org/doc/stable/reference/generated/numpy.arange.html) to create array of radii
* Update the surfaces radius
* [`write_problem`](https://www.montepy.org/en/stable/api/generated/montepy.MCNP_Problem.html#montepy.MCNP_Problem.write_problem) to file

In [ ]:
import numpy as np
MIN_RADIUS = 115.01 # [cm]
MAX_RADIUS = 198.99 # [cm]
CAN_THICKNESS = 1 #[cm]

radii = np.arange(MIN_RADIUS, MAX_RADIUS, 5)
for radius in radii:
    breeder_od.minor_radius = 
    breeder_can_od.minor_radius = 
    problem.write_problem()

# Other Projects you can do
* Add tallies to the model
  * Neutron flux is always needed
    * Can find the neutron attenuation through the materials
    * Examine the neutron energy spectrum changing further as it goes through the various regions
  * Tritium production is essential
  * Nuclear heating is very important for coolant system sizing
  * Radiation damage (displacements per atom) is also important for keeping the structures and magnets superconducting
* Try out different tritium breeder materials
  * Find common tritium breeder materials in the literature
  * Make all of them in the model, and test out different ones in the model
* Look into super conducting magnets more
  * Change magnets from copper to high temp superconductors, e.g., [REBCO](https://doi.org/10.1098/rsta.2018.0354)
  * Play with neutron shielding for the magnets to avoid radiation damage
